In [1]:
import subprocess, sys
_pkgs = ['scipy', 'matplotlib']
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q'] + _pkgs,
    capture_output=True, text=True)
if result.returncode != 0:
    print('pip stderr:', result.stderr[-500:])
import importlib
for _pkg in _pkgs:
    importlib.import_module(_pkg)
    print(f'✅ {_pkg}')
print('Entorno OE3-C listo')


✅ scipy
✅ matplotlib
Entorno OE3-C listo


In [2]:
import sys, os, json, time, importlib.util, subprocess as _sp
sys.path.insert(0, '/srv/python_packages')
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import mlflow

_mlflow_check = _sp.run(["bash", "-c", "curl -s http://localhost:6000/health"],
                        capture_output=True, text=True)
if _mlflow_check.returncode != 0:
    _sp.Popen(["bash", "/srv/start_mlflow.sh"],
              stdout=open("/tmp/mlflow.log", "w"), stderr=_sp.STDOUT)
    time.sleep(5)
    print("MLflow iniciado — http://localhost:6000")
else:
    print("MLflow ya activo — http://localhost:6000")
mlflow.set_tracking_uri('http://localhost:6000')
mlflow.set_experiment('latam2cloud')
print(f'Python  : {sys.version.split()[0]}')
print(f'pandas  : {pd.__version__}')


MLflow ya activo — http://localhost:6000
Python  : 3.11.13
pandas  : 3.0.3


In [3]:
WORK_DIR   = Path('/srv/work')
PRED_PATH  = WORK_DIR / 'predictions_d38.parquet'
OUT_DIR    = WORK_DIR / 'models'

ALERT_LOG_PATH     = OUT_DIR / 'alert_log_d38.parquet'
MTTD_RESULTS_PATH  = OUT_DIR / 'mttd_results_d38.parquet'
DASHBOARD_SPL_PATH = OUT_DIR / 'dashboard_d38.json'
H3_VERDICT_PATH    = OUT_DIR / 'h3_verdict_d38.json'

ROUTE_KEYS  = ['source_agent', 'cloud_provider', 'region']
TS_COL      = 'window'
TARGET_COL  = 'label_ventana'
CLASS_NAMES = ['Normal', 'Baja', 'Media', 'Alta']
WINDOW_MIN  = 30

THR_INCIDENTE   = 0.60
THR_PREVENTIVA  = 0.70
THR_CIERRE      = 0.50
N_WIN_INCIDENTE = 1
N_WIN_PREVENTIVA = 2
N_WIN_CIERRE    = 3

MTTD_BASELINE_MIN = WINDOW_MIN
LOOKBACK_WIN      = 2

ALPHA = 0.05

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('Configuración cargada.')
print(f'Predicciones : {PRED_PATH}')
print(f'Artefactos   : {OUT_DIR}')
print(f'MTTD baseline: {MTTD_BASELINE_MIN} min (= {N_WIN_INCIDENTE} ventana reactiva)')
print(f'α Wilcoxon   : {ALPHA}')


Configuración cargada.
Predicciones : /srv/work/predictions_d38.parquet
Artefactos   : /srv/work/models
MTTD baseline: 30 min (= 1 ventana reactiva)
α Wilcoxon   : 0.05


In [4]:

import pyarrow.parquet as pq

NEEDED_COLS = ROUTE_KEYS + [TS_COL, TARGET_COL,
                             'prob_media', 'prob_alta',
                             'pred_class', 'pred_label']

t0 = time.perf_counter()
df = pq.read_table(PRED_PATH, columns=NEEDED_COLS).to_pandas()
elapsed = time.perf_counter() - t0

df[TS_COL] = pd.to_datetime(df[TS_COL], utc=True)
df = df.sort_values(ROUTE_KEYS + [TS_COL]).reset_index(drop=True)

print(f'Cargado {len(df):,} ventanas en {elapsed:.1f}s')
print(f'Memoria aprox: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB')
print(f'Período: {df[TS_COL].min()} → {df[TS_COL].max()}')
print(f'Rutas únicas: {df[ROUTE_KEYS].drop_duplicates().shape[0]:,}')
print()
print('Distribución label_ventana (test):')
for k, name in enumerate(CLASS_NAMES):
    n = (df[TARGET_COL] == k).sum()
    print(f'  {name:8s} ({k}): {n:>10,}  ({100*n/len(df):5.2f}%)')
print()
print('Distribución pred_class (D38 inferencia):')
for k, name in enumerate(CLASS_NAMES):
    n = (df['pred_label'] == k).sum()
    print(f'  {name:8s} ({k}): {n:>10,}  ({100*n/len(df):5.2f}%)')


Cargado 1,138,360 ventanas en 0.1s
Memoria aprox: 119.8 MB
Período: 2026-09-08 00:00:00+00:00 → 2026-09-30 23:30:00+00:00
Rutas únicas: 1,035

Distribución label_ventana (test):
  Normal   (0):    938,271  (82.42%)
  Baja     (1):    142,039  (12.48%)
  Media    (2):     34,300  ( 3.01%)
  Alta     (3):     23,750  ( 2.09%)

Distribución pred_class (D38 inferencia):
  Normal   (0):    948,681  (83.34%)
  Baja     (1):    136,210  (11.97%)
  Media    (2):     30,268  ( 2.66%)
  Alta     (3):     23,201  ( 2.04%)


In [5]:

DEGRADED_MASK = df[TARGET_COL] >= 2
n_events_total = DEGRADED_MASK.sum()

print(f'Ventanas degradadas (Media+Alta): {n_events_total:,} de {len(df):,} ({100*n_events_total/len(df):.2f}%)')

def count_events(grp):
    """Cuenta tramos contiguos de label_ventana >= 2 en el grupo."""
    is_deg = (grp[TARGET_COL] >= 2).values
    events = 0
    in_event = False
    for v in is_deg:
        if v and not in_event:
            events += 1
            in_event = True
        elif not v:
            in_event = False
    return events

event_counts = df.groupby(ROUTE_KEYS).apply(count_events)
total_events = event_counts.sum()
routes_with_events = (event_counts > 0).sum()

print(f'Eventos de degradación totales   : {total_events:,}')
print(f'Rutas con ≥1 evento              : {routes_with_events:,} de {len(event_counts):,}')
print(f'Eventos promedio por ruta activa : {total_events / max(routes_with_events, 1):.1f}')


Ventanas degradadas (Media+Alta): 58,050 de 1,138,360 (5.10%)
Eventos de degradación totales   : 22,716
Rutas con ≥1 evento              : 760 de 1,035
Eventos promedio por ruta activa : 29.9


In [6]:

def apply_alert_rules(grp):
    """
    Aplica reglas §10.7 a un grupo (ruta) ordenado por ventana.
    Devuelve lista de dicts con cada alerta disparada.
    """
    grp = grp.reset_index(drop=True)
    n = len(grp)
    alerts = []

    p_ma = (grp['prob_media'] + grp['prob_alta']).values
    p_a  = grp['prob_alta'].values

    for t in range(n):
        route_info = {k: grp.loc[t, k] for k in ROUTE_KEYS}
        ts = grp.loc[t, TS_COL]
        true_label = grp.loc[t, TARGET_COL]

        if p_a[t] >= THR_INCIDENTE:
            alerts.append({**route_info,
                           'window': ts, 'alert_type': 'INCIDENTE',
                           'prob_alta': float(p_a[t]),
                           'prob_media_alta': float(p_ma[t]),
                           'true_label': int(true_label)})

        elif t >= 1 and p_ma[t] >= THR_PREVENTIVA and p_ma[t-1] >= THR_PREVENTIVA:
            alerts.append({**route_info,
                           'window': ts, 'alert_type': 'PREVENTIVA',
                           'prob_alta': float(p_a[t]),
                           'prob_media_alta': float(p_ma[t]),
                           'true_label': int(true_label)})

        if t >= 2 and p_ma[t] < THR_CIERRE and p_ma[t-1] < THR_CIERRE and p_ma[t-2] < THR_CIERRE:
            alerts.append({**route_info,
                           'window': ts, 'alert_type': 'CIERRE',
                           'prob_alta': float(p_a[t]),
                           'prob_media_alta': float(p_ma[t]),
                           'true_label': int(true_label)})

    return alerts

print('Aplicando reglas §10.7 a todas las rutas...')
t0 = time.perf_counter()
all_alerts = []
for route_key, grp in df.groupby(ROUTE_KEYS, sort=False):
    all_alerts.extend(apply_alert_rules(grp))

alert_df = pd.DataFrame(all_alerts)
alert_df['window'] = pd.to_datetime(alert_df['window'], utc=True)
elapsed = time.perf_counter() - t0

print(f'Procesado en {elapsed:.1f}s')
print(f'Total alertas generadas: {len(alert_df):,}')
print()
for atype in ['INCIDENTE', 'PREVENTIVA', 'CIERRE']:
    n = (alert_df['alert_type'] == atype).sum()
    print(f'  {atype:12s}: {n:,}')

active_alerts = alert_df[alert_df['alert_type'].isin(['INCIDENTE', 'PREVENTIVA'])]
tp = (active_alerts['true_label'] >= 2).sum()
total_active = len(active_alerts)
precision = tp / max(total_active, 1)
print(f'\nPrecisión alertas activas (TP/total): {tp:,}/{total_active:,} = {precision:.3f}')


Aplicando reglas §10.7 a todas las rutas...
Procesado en 195.8s
Total alertas generadas: 1,087,476

  INCIDENTE   : 8,596
  PREVENTIVA  : 15,352
  CIERRE      : 1,063,528

Precisión alertas activas (TP/total): 22,861/23,948 = 0.955


In [7]:
alert_df.to_parquet(ALERT_LOG_PATH, index=False)
print(f'alert_log_d38.parquet guardado: {ALERT_LOG_PATH.stat().st_size / 1e3:.1f} KB')
print(f'Columnas: {list(alert_df.columns)}')
print(alert_df.head(3).to_string())


alert_log_d38.parquet guardado: 15211.7 KB
Columnas: ['source_agent', 'cloud_provider', 'region', 'window', 'alert_type', 'prob_alta', 'prob_media_alta', 'true_label']
                   source_agent cloud_provider              region                    window alert_type  prob_alta  prob_media_alta  true_label
0  Belo Horizonte, Brazil (TIM)            AWS  Chicago-Local-Zone 2026-09-08 01:00:00+00:00     CIERRE   0.053871         0.096765           0
1  Belo Horizonte, Brazil (TIM)            AWS  Chicago-Local-Zone 2026-09-08 01:30:00+00:00     CIERRE   0.074806         0.087473           0
2  Belo Horizonte, Brazil (TIM)            AWS  Chicago-Local-Zone 2026-09-08 02:00:00+00:00     CIERRE   0.055858         0.078439           0


In [8]:

lookback_td = pd.Timedelta(minutes=WINDOW_MIN * LOOKBACK_WIN)

active_alerts_df = alert_df[alert_df['alert_type'].isin(['INCIDENTE', 'PREVENTIVA'])].copy()
active_alerts_df['window'] = pd.to_datetime(active_alerts_df['window'], utc=True)

alert_grouped = {}
for rk, grp_a in active_alerts_df.groupby(ROUTE_KEYS, sort=False):
    alert_grouped[rk] = sorted(grp_a['window'].tolist())

print(f'Rutas con alertas activas: {len(alert_grouped):,}')
print(f'Total alertas indexadas  : {sum(len(v) for v in alert_grouped.values()):,}')

def find_first_alert_for_event(event_start, event_end, sorted_alert_ts):
    """Busca la primera alerta en [event_start - lookback, event_end]."""
    search_from = event_start - lookback_td
    for ts in sorted_alert_ts:
        if ts > event_end:
            break
        if ts >= search_from:
            return ts
    return None

print('Calculando MTTD por evento y ruta...')
t0 = time.perf_counter()
mttd_rows = []

for route_key, grp in df.groupby(ROUTE_KEYS, sort=False):
    grp = grp.sort_values(TS_COL).reset_index(drop=True)
    is_deg = (grp[TARGET_COL] >= 2).values
    n = len(grp)
    route_alert_ts = alert_grouped.get(route_key, [])
    route_dict = dict(zip(ROUTE_KEYS, route_key))
    i = 0
    while i < n:
        if is_deg[i]:
            j = i
            while j < n and is_deg[j]:
                j += 1
            event_start = grp.loc[i, TS_COL]
            event_end   = grp.loc[j - 1, TS_COL]
            event_len   = j - i
            first_alert = find_first_alert_for_event(event_start, event_end, route_alert_ts)
            mttd_ml = float((first_alert - event_start).total_seconds() / 60) \
                      if first_alert is not None else None
            mttd_rows.append({**route_dict,
                               'event_start':       event_start,
                               'event_end':         event_end,
                               'event_len_windows': event_len,
                               'mttd_ml_min':       mttd_ml,
                               'mttd_baseline_min': float(MTTD_BASELINE_MIN),
                               'alerted':           first_alert is not None})
            i = j
        else:
            i += 1

mttd_df = pd.DataFrame(mttd_rows)
elapsed = time.perf_counter() - t0

print(f'Procesado en {elapsed:.1f}s')
print(f'Eventos totales             : {len(mttd_df):,}')
print(f'Eventos alertados (ML)      : {mttd_df["alerted"].sum():,} ({100*mttd_df["alerted"].mean():.1f}%)')
print(f'Eventos NO alertados (missed): {(~mttd_df["alerted"]).sum():,}')


Rutas con alertas activas: 365
Total alertas indexadas  : 23,948
Calculando MTTD por evento y ruta...
Procesado en 4.2s
Eventos totales             : 22,716
Eventos alertados (ML)      : 3,785 (16.7%)
Eventos NO alertados (missed): 18,931


In [9]:

paired = mttd_df[mttd_df['alerted']].dropna(subset=['mttd_ml_min'])
n_pairs = len(paired)

mttd_ml_arr       = paired['mttd_ml_min'].values.astype(float)
mttd_baseline_arr = paired['mttd_baseline_min'].values.astype(float)
diff = mttd_baseline_arr - mttd_ml_arr

print(f'Pares para test Wilcoxon: {n_pairs:,}')
print(f'MTTD_ML       — media: {mttd_ml_arr.mean():.1f} min | mediana: {np.median(mttd_ml_arr):.1f} min')
print(f'MTTD_baseline — media: {mttd_baseline_arr.mean():.1f} min | mediana: {np.median(mttd_baseline_arr):.1f} min')
print(f'Diferencia    — media: {diff.mean():.1f} min | mediana: {np.median(diff):.1f} min')
print(f'  (positivo = ML detectó antes que baseline)')
print()

if n_pairs >= 10:
    stat, p_value = wilcoxon(diff, alternative='greater')
    print(f'Wilcoxon signed-rank test (alternative="greater"):')
    print(f'  W = {stat:.1f}')
    print(f'  p = {p_value:.6f}')
    print()
    if p_value < ALPHA:
        h3_verdict = 'VERIFICADA'
        print(f'✅ H3 VERIFICADA: p={p_value:.4f} < α={ALPHA}')
        print('   El sistema ML reduce significativamente el MTTD vs monitoreo reactivo.')
    else:
        h3_verdict = 'NO_VERIFICADA'
        print(f'⚠️  H3 NO VERIFICADA: p={p_value:.4f} ≥ α={ALPHA}')
        print('   La diferencia de MTTD no es estadísticamente significativa.')
else:
    stat, p_value = None, None
    h3_verdict = 'INSUFICIENTE'
    print(f'⚠️  Pares insuficientes ({n_pairs} < 10) para test estadístico.')
    print('   H3: INSUFICIENTE — requiere más datos de producción.')

n_faster  = (diff > 0).sum()
n_equal   = (diff == 0).sum()
n_slower  = (diff < 0).sum()
print()
print(f'ML más rápido que baseline : {n_faster:,} ({100*n_faster/max(n_pairs,1):.1f}%)')
print(f'ML igual que baseline      : {n_equal:,} ({100*n_equal/max(n_pairs,1):.1f}%)')
print(f'ML más lento que baseline  : {n_slower:,} ({100*n_slower/max(n_pairs,1):.1f}%)')


Pares para test Wilcoxon: 3,785
MTTD_ML       — media: 8.7 min | mediana: 0.0 min
MTTD_baseline — media: 30.0 min | mediana: 30.0 min
Diferencia    — media: 21.3 min | mediana: 30.0 min
  (positivo = ML detectó antes que baseline)

Wilcoxon signed-rank test (alternative="greater"):
  W = 3933850.5
  p = 0.000000

✅ H3 VERIFICADA: p=0.0000 < α=0.05
   El sistema ML reduce significativamente el MTTD vs monitoreo reactivo.

ML más rápido que baseline : 1,950 (51.5%)
ML igual que baseline      : 563 (14.9%)
ML más lento que baseline  : 1,272 (33.6%)


In [10]:
mttd_df.to_parquet(MTTD_RESULTS_PATH, index=False)
print(f'mttd_results_d38.parquet guardado: {MTTD_RESULTS_PATH.stat().st_size / 1e3:.1f} KB')

class _NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.bool_,)):   return bool(o)
        if isinstance(o, np.integer):    return int(o)
        if isinstance(o, np.floating):   return float(o)
        return super().default(o)

h3_record = {
    'h3_verdict': h3_verdict,
    'alpha': ALPHA,
    'n_events_total': int(len(mttd_df)),
    'n_events_alerted': int(mttd_df['alerted'].sum()),
    'n_pairs_wilcoxon': int(n_pairs),
    'wilcoxon_stat': float(stat) if stat is not None else None,
    'wilcoxon_p': float(p_value) if p_value is not None else None,
    'mttd_ml_mean_min': float(mttd_ml_arr.mean()) if n_pairs > 0 else None,
    'mttd_ml_median_min': float(np.median(mttd_ml_arr)) if n_pairs > 0 else None,
    'mttd_baseline_min': float(MTTD_BASELINE_MIN),
    'mttd_reduction_mean_min': float(diff.mean()) if n_pairs > 0 else None,
    'alert_precision': float(precision),
    'model_run_id_nb09': '4e0516d4cb3743149e905eea67dc87e8',
    'thresholds': {
        'incidente': THR_INCIDENTE,
        'preventiva': THR_PREVENTIVA,
        'cierre': THR_CIERRE,
    }
}

with open(H3_VERDICT_PATH, 'w') as f:
    json.dump(h3_record, f, indent=2, cls=_NpEncoder)
print(f'h3_verdict_d38.json guardado: {H3_VERDICT_PATH.stat().st_size / 1e3:.1f} KB')
print(json.dumps(h3_record, indent=2))


mttd_results_d38.parquet guardado: 106.7 KB
h3_verdict_d38.json guardado: 0.5 KB
{
  "h3_verdict": "VERIFICADA",
  "alpha": 0.05,
  "n_events_total": 22716,
  "n_events_alerted": 3785,
  "n_pairs_wilcoxon": 3785,
  "wilcoxon_stat": 3933850.5,
  "wilcoxon_p": 1.3689414667543746e-148,
  "mttd_ml_mean_min": 8.694848084544253,
  "mttd_ml_median_min": 0.0,
  "mttd_baseline_min": 30.0,
  "mttd_reduction_mean_min": 21.305151915455745,
  "alert_precision": 0.9546099883080007,
  "model_run_id_nb09": "4e0516d4cb3743149e905eea67dc87e8",
  "thresholds": {
    "incidente": 0.6,
    "preventiva": 0.7,
    "cierre": 0.5
  }
}


In [11]:

CLASS_COLORS = {
    'Normal': '#2ecc71',
    'Baja':   '#f39c12',
    'Media':  '#e67e22',
    'Alta':   '#e74c3c',
}
ALERT_COLORS = {
    'PREVENTIVA': '#f39c12',
    'INCIDENTE':  '#e74c3c',
    'CIERRE':     '#2ecc71',
}

_pred_cols = ROUTE_KEYS + [TS_COL, 'pred_class', 'pred_label', TARGET_COL]
_df_pred = pd.read_parquet(PRED_PATH, columns=_pred_cols)
_df_pred[TS_COL] = pd.to_datetime(_df_pred[TS_COL], utc=True)

fig, axes = plt.subplots(3, 1, figsize=(14, 14))
fig.suptitle('Dashboard LATAM2Cloud — Ensemble D38 · Test Set Sep 8–30, 2026',
             fontsize=14, fontweight='bold', y=0.98)

ax1 = axes[0]
_df_pred['date'] = _df_pred[TS_COL].dt.date
daily = _df_pred.groupby(['date', 'pred_class']).size().unstack(fill_value=0)
for cls in CLASS_NAMES:
    if cls not in daily.columns:
        daily[cls] = 0
daily = daily[CLASS_NAMES]

daily_pct = daily.div(daily.sum(axis=1), axis=0) * 100
dates = [str(d) for d in daily_pct.index]
x = range(len(dates))

bottom = np.zeros(len(dates))
for cls in CLASS_NAMES:
    vals = daily_pct[cls].values
    ax1.bar(x, vals, bottom=bottom, label=cls,
            color=CLASS_COLORS[cls], alpha=0.85, width=0.8)
    bottom += vals

ax1.set_title('Panel 1 — Distribución Diaria de Predicciones por Clase', fontweight='bold')
ax1.set_ylabel('% Ventanas')
ax1.set_xticks(list(x)[::3])
ax1.set_xticklabels(dates[::3], rotation=45, ha='right', fontsize=8)
ax1.legend(loc='upper right', fontsize=8)
ax1.set_ylim(0, 100)

ax2 = axes[1]
pred_counts  = _df_pred['pred_class'].value_counts().reindex(CLASS_NAMES, fill_value=0)
true_counts  = _df_pred[TARGET_COL].map(dict(enumerate(CLASS_NAMES))).value_counts().reindex(CLASS_NAMES, fill_value=0)

xpos  = np.arange(len(CLASS_NAMES))
width = 0.38
bars1 = ax2.bar(xpos - width/2, pred_counts.values / 1e3, width=width,
                color=[CLASS_COLORS[c] for c in CLASS_NAMES], alpha=0.85, label='Predicho')
bars2 = ax2.bar(xpos + width/2, true_counts.values / 1e3, width=width,
                color=[CLASS_COLORS[c] for c in CLASS_NAMES], alpha=0.45,
                edgecolor='black', linewidth=0.7, label='Real')

ax2.set_title('Panel 2 — Predicho vs Real (Test Set)', fontweight='bold')
ax2.set_ylabel('Ventanas (miles)')
ax2.set_xticks(xpos)
ax2.set_xticklabels(CLASS_NAMES)
ax2.legend(fontsize=9)
for bar in bars1:
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             f'{bar.get_height():.0f}k', ha='center', va='bottom', fontsize=7)

ax3 = axes[2]
if len(alert_df) > 0:
    alert_df['date'] = pd.to_datetime(alert_df['window'], utc=True).dt.date
    alert_daily = alert_df.groupby(['date', 'alert_type']).size().unstack(fill_value=0)
    for atype in ['PREVENTIVA', 'INCIDENTE', 'CIERRE']:
        if atype not in alert_daily.columns:
            alert_daily[atype] = 0
    alert_dates = [str(d) for d in alert_daily.index]
    ax3_x = range(len(alert_dates))
    bottom3 = np.zeros(len(alert_dates))
    for atype in ['INCIDENTE', 'PREVENTIVA', 'CIERRE']:
        vals = alert_daily[atype].values if atype in alert_daily.columns else np.zeros(len(alert_dates))
        ax3.bar(ax3_x, vals, bottom=bottom3, label=atype,
                color=ALERT_COLORS[atype], alpha=0.85, width=0.8)
        bottom3 += vals
    ax3.set_xticks(list(ax3_x)[::3])
    ax3.set_xticklabels(alert_dates[::3], rotation=45, ha='right', fontsize=8)
else:
    ax3.text(0.5, 0.5, 'Sin alertas generadas', transform=ax3.transAxes,
             ha='center', va='center', fontsize=12)

ax3.set_title('Panel 3 — Historial de Alertas §10.7 (diario)', fontweight='bold')
ax3.set_ylabel('Alertas')
ax3.legend(fontsize=9)

plt.tight_layout(rect=[0, 0, 1, 0.97])
DASH_PNG = OUT_DIR / 'dashboard_d38.png'
fig.savefig(DASH_PNG, dpi=120, bbox_inches='tight')
plt.close(fig)
print(f'Dashboard guardado: {DASH_PNG}  ({DASH_PNG.stat().st_size / 1e3:.0f} KB)')


Dashboard guardado: /srv/work/models/dashboard_d38.png  (108 KB)


In [12]:

SPL_QUERIES = {
    "panel1_daily_class_distribution": {
        "title": "Panel 1 — Distribución Diaria de Clases Predichas",
        "description": "Stacked area chart: distribución de pred_class por día",
        "spl": (
            "index=dsdl sourcetype=latam2cloud_ml "
            "| eval date=strftime(_time, \"%Y-%m-%d\") "
            "| stats count by date, pred_class "
            "| eval order=case(pred_class==\"Normal\",0, pred_class==\"Baja\",1, "
            "                  pred_class==\"Media\",2, pred_class==\"Alta\",3, true(),4) "
            "| sort date, order "
            "| timechart span=1d count by pred_class"
        ),
        "visualization": "charting.chart=bar charting.chart.stackMode=stacked"
    },
    "panel2_class_counts": {
        "title": "Panel 2 — Recuento de Predicciones por Clase",
        "description": "Barras verticales: pred_class vs label_ventana (real)",
        "spl": (
            "index=dsdl sourcetype=latam2cloud_ml "
            "| stats count as predicho by pred_class "
            "| join type=outer pred_class "
            "    [search index=dsdl sourcetype=latam2cloud_ml "
            "     | eval real_class=case(label_ventana==0,\"Normal\", label_ventana==1,\"Baja\", "
            "                           label_ventana==2,\"Media\", label_ventana==3,\"Alta\") "
            "     | stats count as real by real_class "
            "     | rename real_class as pred_class] "
            "| table pred_class predicho real"
        ),
        "visualization": "charting.chart=bar"
    },
    "panel3_alert_history": {
        "title": "Panel 3 — Historial de Alertas §10.7",
        "description": "Timeline de alertas INCIDENTE / PREVENTIVA / CIERRE",
        "spl": (
            "index=dsdl sourcetype=latam2cloud_alerts "
            "| timechart span=1h count by alert_type "
            "| rename INCIDENTE as \"Incidente\", PREVENTIVA as \"Preventiva\", CIERRE as \"Cierre\""
        ),
        "visualization": "charting.chart=area charting.chart.stackMode=stacked"
    },
    "panel4_mttd_comparison": {
        "title": "Panel 4 (Adicional) — MTTD_ML vs MTTD_baseline por Evento",
        "description": "Scatter de MTTD_ML vs MTTD_baseline para evidencia H3",
        "spl": (
            "| inputlookup mttd_results_d38.csv "
            "| where alerted=1 "
            "| table event_start source_agent cloud_provider region "
            "        mttd_ml_min mttd_baseline_min event_len_windows "
            "| eval mttd_reduction=mttd_baseline_min-mttd_ml_min"
        ),
        "visualization": "charting.chart=scatter"
    }
}

with open(DASHBOARD_SPL_PATH, 'w') as f:
    json.dump(SPL_QUERIES, f, indent=2)
print(f'dashboard_d38.json guardado: {DASHBOARD_SPL_PATH.stat().st_size / 1e3:.1f} KB')
for k, v in SPL_QUERIES.items():
    print(f'  {k}: {v["title"]}')


dashboard_d38.json guardado: 2.2 KB
  panel1_daily_class_distribution: Panel 1 — Distribución Diaria de Clases Predichas
  panel2_class_counts: Panel 2 — Recuento de Predicciones por Clase
  panel3_alert_history: Panel 3 — Historial de Alertas §10.7
  panel4_mttd_comparison: Panel 4 (Adicional) — MTTD_ML vs MTTD_baseline por Evento


In [13]:

with mlflow.start_run(run_name='nb10_dashboard_mttd_h3') as run:
    run_id = run.info.run_id

    mlflow.log_params({
        'notebook': '10 splunk_dashboard v1',
        'stage': 'OE3-C',
        'input_parquet': 'predictions_d38.parquet',
        'n_ventanas': len(df),
        'thr_incidente': THR_INCIDENTE,
        'thr_preventiva': THR_PREVENTIVA,
        'thr_cierre': THR_CIERRE,
        'window_min': WINDOW_MIN,
        'mttd_baseline_min': MTTD_BASELINE_MIN,
        'alpha': ALPHA,
    })

    metrics = {
        'n_events_total': int(len(mttd_df)),
        'n_events_alerted': int(mttd_df['alerted'].sum()),
        'alert_precision': float(precision),
        'n_alerts_incidente': int((alert_df['alert_type'] == 'INCIDENTE').sum()),
        'n_alerts_preventiva': int((alert_df['alert_type'] == 'PREVENTIVA').sum()),
        'n_alerts_cierre': int((alert_df['alert_type'] == 'CIERRE').sum()),
    }
    if n_pairs > 0:
        metrics.update({
            'mttd_ml_mean_min':      float(mttd_ml_arr.mean()),
            'mttd_ml_median_min':    float(np.median(mttd_ml_arr)),
            'mttd_baseline_min':     float(MTTD_BASELINE_MIN),
            'mttd_reduction_mean':   float(diff.mean()),
            'mttd_reduction_median': float(np.median(diff)),
            'n_pairs_wilcoxon':      int(n_pairs),
        })
        if stat is not None:
            metrics['wilcoxon_stat'] = float(stat)
            metrics['wilcoxon_p']    = float(p_value)

    mlflow.log_metrics(metrics)
    mlflow.set_tag('h3_verdict', h3_verdict)

    mlflow.log_artifact(str(ALERT_LOG_PATH))
    mlflow.log_artifact(str(MTTD_RESULTS_PATH))
    mlflow.log_artifact(str(DASHBOARD_SPL_PATH))
    mlflow.log_artifact(str(H3_VERDICT_PATH))
    mlflow.log_artifact(str(DASH_PNG))

    print(f'MLflow run_id: {run_id}')
    print(f'H3 verdict   : {h3_verdict}')
    print(f'Métricas MTTD registradas.')
    print(f'Artefactos:')
    for p in [ALERT_LOG_PATH, MTTD_RESULTS_PATH, DASHBOARD_SPL_PATH, H3_VERDICT_PATH, DASH_PNG]:
        print(f'  {p.name}')


MLflow run_id: a020ca029f454be6b1e1e1e67a6be59c
H3 verdict   : VERIFICADA
Métricas MTTD registradas.
Artefactos:
  alert_log_d38.parquet
  mttd_results_d38.parquet
  dashboard_d38.json
  h3_verdict_d38.json
  dashboard_d38.png
🏃 View run nb10_dashboard_mttd_h3 at: http://localhost:6000/#/experiments/681062908553326840/runs/a020ca029f454be6b1e1e1e67a6be59c
🧪 View experiment at: http://localhost:6000/#/experiments/681062908553326840
